# Adult Census Income — Iteration 2 Notebook (Six-Step Coursework Headings)

This notebook reorganizes the existing Iteration 1 coding workflow under the exact six coursework brief step headings.


## Legacy Note

This notebook is **legacy (Iteration 2)**.

Canonical working notebook is:
- `notebooks/adult_census_income_iteration3_six_steps.ipynb`


## 1. Obtain a dataset and frame the predictive problem


### Code Chunk 1: Environment and path setup

**What this does:** Imports libraries, sets project/data/output paths, and ensures required directories exist before running anything else.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Environment and path setup
# Notes: Imports libraries, sets project/data/output paths, and ensures required directories exist before running anything else.
# Attribution: [Modified-from-agent]

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

ROOT = Path.cwd().resolve()
if not (ROOT / 'data/raw/adult_census_income/adult.csv').exists() and (ROOT.parent / 'data/raw/adult_census_income/adult.csv').exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / 'data/raw/adult_census_income/adult.csv'
PROC_DIR = ROOT / 'data/processed/adult_census_income'
OUT_DIR = ROOT / 'outputs/adult_census_income'
FIG_DIR = OUT_DIR / 'figures'
METRICS_DIR = OUT_DIR / 'metrics'
MODELS_DIR = OUT_DIR / 'models'

for d in [PROC_DIR, FIG_DIR, METRICS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert RAW_PATH.exists(), f'Missing dataset: {RAW_PATH}'
print('ROOT:', ROOT)
print('RAW_PATH:', RAW_PATH)


### Code Chunk 2: Problem framing metadata

**What this does:** Defines target/stakeholder/success criteria and writes framing artefacts for the project record.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Problem framing metadata
# Notes: Defines target/stakeholder/success criteria and writes framing artefacts for the project record.
# Attribution: [Modified-from-agent]

framing = {
    'iteration': 'iteration_1_module_1',
    'dataset': 'adult_census_income',
    'target': 'income',
    'stakeholder': 'workforce policy / HR analytics team',
    'success_criteria_for_next_module': {
        'f1_weighted_min': 0.82,
        'positive_recall_min': 0.75,
        'calibration_required': True
    },
    'constraints': [
        'Interpretability is mandatory',
        'Fairness-sensitive attributes require explicit later audit'
    ],
    'assumptions': [
        'Historical records are representative enough for initial modelling',
        'Labels and sampling may contain bias; outputs are decision support'
    ]
}

problem_statement_md = f'''# Problem Statement: adult_census_income

## Iteration
- {framing['iteration']}

## Target
- `{framing['target']}`

## Stakeholder
- {framing['stakeholder']}

## Success Criteria (for Module 2)
- f1_weighted >= {framing['success_criteria_for_next_module']['f1_weighted_min']}
- positive-class recall >= {framing['success_criteria_for_next_module']['positive_recall_min']}
- calibrated probabilities required for threshold policy

## Constraints
- Interpretability required
- Fairness-sensitive fields must be audited

## Assumptions
- Initial distribution is sufficiently representative
- Potential label/sampling bias is acknowledged
'''
(OUT_DIR / 'problem_statement.md').write_text(problem_statement_md, encoding='utf-8')

print('Wrote:', OUT_DIR / 'problem_statement.md')
print(json.dumps(framing, indent=2))


### Column Meaning Notes (Ambiguous Features)


### Ambiguous Column Meanings

Some Adult dataset columns are not self-explanatory. This reference clarifies them before EDA/modelling.

| Column | Meaning | Practical Modelling Note |
|---|---|---|
| `fnlwgt` | **Final sample weight** from Census processing. It approximates how many people in the population this record represents. | It is not a direct socio-economic attribute. Use with caution as a feature; also consider sensitivity checks with and without it. |
| `education.num` | Numeric encoding of education level (ordered), corresponding to `education` text categories. | It is ordinal. Avoid treating it as unrelated categories. Also check redundancy with `education` to prevent duplicated signal. |
| `capital.gain` | Reported capital gains (profit from sale of assets/investments) for the reference period. | Highly right-skewed with many zeros; robust scaling and careful outlier handling are appropriate. |
| `capital.loss` | Reported capital losses (loss from sale of assets/investments) for the reference period. | Also sparse/right-skewed; likely interacts with income outcome and should be assessed for leverage effects. |

Interpretation caveat: values reflect survey/administrative coding decisions, not perfect economic measurements.


## 2. Explore the data to gain insights


### Code Chunk 3: Load and normalize raw data

**What this does:** Loads Adult CSV, standardizes categorical strings, and converts missing-value proxies (`?`) to nulls.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Load and normalize raw data
# Notes: Loads Adult CSV, standardizes categorical strings, and converts missing-value proxies (`?`) to nulls.
# Attribution: [Modified-from-agent]

df = pd.read_csv(RAW_PATH)
obj_cols = df.select_dtypes(include=['object', 'string']).columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace('?', np.nan)

assert 'income' in df.columns
print('shape:', df.shape)
print('target distribution:')
print(df['income'].value_counts(dropna=False))
df.head(3)


### Code Chunk 4: EDA: target distribution

**What this does:** Computes class distribution, plots imbalance, and saves the target-distribution figure.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: EDA: target distribution
# Notes: Computes class distribution, plots imbalance, and saves the target-distribution figure.
# Attribution: [Modified-from-agent]

vc = df['income'].value_counts(dropna=False)
imbalance_ratio = float(vc.max() / vc.min())
plt.figure(figsize=(6,4))
vc.plot(kind='bar', color=['#4C78A8', '#F58518'])
plt.title('Target Distribution: Income (Class Imbalance Present)')
plt.ylabel('Count')
plt.tight_layout()
target_path = FIG_DIR / 'target_distribution.png'
plt.savefig(target_path, dpi=150)
plt.show()
print('imbalance_ratio:', round(imbalance_ratio, 3))
print('saved:', target_path)


### Code Chunk 5: EDA: numeric distributions

**What this does:** Plots key numeric feature distributions to inspect skewness/outliers and saves the figure.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: EDA: numeric distributions
# Notes: Plots key numeric feature distributions to inspect skewness/outliers and saves the figure.
# Attribution: [Modified-from-agent]

num_cols = [c for c in ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week'] if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for i, c in enumerate(num_cols):
    axes[i].hist(df[c].dropna(), bins=40, color='#4C78A8', alpha=0.9)
    axes[i].set_title(c)
for i in range(len(num_cols), len(axes)):
    axes[i].axis('off')
fig.suptitle('Numeric Feature Distributions (Outlier-Heavy in Gain/Loss)', y=1.02)
fig.tight_layout()
numeric_path = FIG_DIR / 'numeric_distributions.png'
fig.savefig(numeric_path, dpi=150)
plt.show()
print('saved:', numeric_path)


### Code Chunk 6: EDA: categorical cardinality

**What this does:** Computes categorical cardinalities to anticipate encoding complexity and inference-time unknowns.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: EDA: categorical cardinality
# Notes: Computes categorical cardinalities to anticipate encoding complexity and inference-time unknowns.
# Attribution: [Modified-from-agent]

cat_cols = [c for c in ['workclass','education','occupation','relationship','race','sex','native.country'] if c in df.columns]
cardinalities = pd.Series({c: df[c].nunique(dropna=True) for c in cat_cols}).sort_values(ascending=False)
plt.figure(figsize=(8,4))
cardinalities.plot(kind='bar', color='#54A24B')
plt.title('Categorical Cardinality (High Cardinality in native.country/occupation)')
plt.ylabel('Unique values')
plt.tight_layout()
cat_path = FIG_DIR / 'categorical_cardinality.png'
plt.savefig(cat_path, dpi=150)
plt.show()
print('saved:', cat_path)


### Code Chunk 7: EDA: missingness summary

**What this does:** Quantifies missingness percentages and visualizes top affected columns.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: EDA: missingness summary
# Notes: Quantifies missingness percentages and visualizes top affected columns.
# Attribution: [Modified-from-agent]

miss = (df.isna().mean() * 100).sort_values(ascending=False)
plt.figure(figsize=(8,4))
miss.head(10).plot(kind='bar', color='#E45756')
plt.title('Top Missingness Columns (Imputation Required)')
plt.ylabel('Missingness (%)')
plt.tight_layout()
miss_path = FIG_DIR / 'missingness_summary.png'
plt.savefig(miss_path, dpi=150)
plt.show()
print('saved:', miss_path)
miss.head(8)


### Code Chunk 8: EDA: feature-target signal check

**What this does:** Builds a quick numeric separation proxy against target classes to guide feature retention decisions.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: EDA: feature-target signal check
# Notes: Builds a quick numeric separation proxy against target classes to guide feature retention decisions.
# Attribution: [Modified-from-agent]

target_map = {k: i for i, k in enumerate(sorted(df['income'].dropna().unique()))}
y_num = df['income'].map(target_map)
assoc = {}
for c in num_cols:
    a = df[[c]].copy()
    a['y'] = y_num
    g = a.groupby('y')[c].median()
    if len(g) >= 2:
        assoc[c] = float(abs(g.iloc[-1] - g.iloc[0]))
assoc = pd.Series(assoc).sort_values(ascending=False)
plt.figure(figsize=(8,4))
assoc.plot(kind='bar', color='#B279A2')
plt.title('Numeric Feature-Target Separation (Median Gap)')
plt.ylabel('Absolute median difference')
plt.tight_layout()
ft_path = FIG_DIR / 'feature_target_checks.png'
plt.savefig(ft_path, dpi=150)
plt.show()
print('saved:', ft_path)
assoc


### Code Chunk 8B: EDA leakage, correlation, bivariate, and outlier diagnostics

**What this does:** Adds high-value diagnostics required by the rubric: leakage-suspicion chart with threshold, numeric correlation heatmap, bivariate target plots, and explicit outlier policy evidence.

**How to use:** Run after the EDA feature-target signal cell. It writes additional figures to `outputs/adult_census_income/figures/`.


In [ ]:
# Cell purpose: EDA enhancement for leakage/correlation/bivariate/outlier diagnostics
# Notes: Adds rubric-targeted visuals and explicit outlier summary used for methodology justification.
# Attribution: [Modified-from-agent]

from sklearn.ensemble import IsolationForest

# Leakage suspicion chart (numeric feature vs binary target correlation)
y_bin_eda = df['income'].map({'<=50K': 0, '>50K': 1})
num_df_eda = df.select_dtypes(include=['number']).copy()
num_target_corr = num_df_eda.apply(lambda s: s.corr(y_bin_eda)).fillna(0.0)
num_target_corr_abs = num_target_corr.abs().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(num_target_corr_abs.index, num_target_corr_abs.values, color='#4C78A8')
plt.axhline(0.9, color='red', linestyle='--', linewidth=1.5, label='Suspicion threshold (0.9)')
plt.xticks(rotation=35, ha='right')
plt.ylabel('Absolute correlation with target')
plt.title('Leakage Suspicion Check: Numeric Features vs Income')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'leakage_suspicion_numeric_corr.png', dpi=150)
plt.show()

suspicious_features = num_target_corr_abs[num_target_corr_abs >= 0.9].index.tolist()
print('Potential leakage-suspicious numeric features (>=0.9):', suspicious_features if suspicious_features else 'None')

# Numeric correlation heatmap
corr_mat = num_df_eda.corr().fillna(0.0)
plt.figure(figsize=(7, 6))
im = plt.imshow(corr_mat.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr_mat.columns)), corr_mat.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_mat.index)), corr_mat.index)
plt.title('Numeric Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'numeric_correlation_heatmap.png', dpi=150)
plt.show()

# Bivariate feature-target plots
biv_features = [c for c in ['age', 'hours.per.week', 'capital.gain', 'capital.loss'] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for i, feat in enumerate(biv_features):
    df.boxplot(column=feat, by='income', ax=axes[i])
    axes[i].set_title(f'{feat} by income class')
    axes[i].set_xlabel('income')
for j in range(len(biv_features), len(axes)):
    axes[j].axis('off')
plt.suptitle('Bivariate Diagnostics: Feature vs Target')
plt.tight_layout()
plt.savefig(FIG_DIR / 'bivariate_boxplots_by_target.png', dpi=150)
plt.show()

# Outlier diagnostics: IQR outlier rates + IsolationForest scatter
outlier_rates = {}
for feat in biv_features:
    q1 = df[feat].quantile(0.25)
    q3 = df[feat].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[feat] < lower) | (df[feat] > upper)
    outlier_rates[feat] = float(mask.mean() * 100)

outlier_rate_s = pd.Series(outlier_rates).sort_values(ascending=False)
plt.figure(figsize=(7, 4))
plt.bar(outlier_rate_s.index, outlier_rate_s.values, color='#E45756')
plt.ylabel('Outlier rate (%) by IQR rule')
plt.title('Outlier Policy Support: IQR Outlier Rates')
plt.tight_layout()
plt.savefig(FIG_DIR / 'outlier_iqr_rates.png', dpi=150)
plt.show()

iso_feats = [c for c in ['age', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week'] if c in df.columns]
iso_df = df[iso_feats].copy().fillna(df[iso_feats].median())
iso = IsolationForest(contamination=0.03, random_state=42)
iso_flag = (iso.fit_predict(iso_df) == -1).astype(int)

if {'age', 'hours.per.week'}.issubset(df.columns):
    plt.figure(figsize=(7, 5))
    plt.scatter(df['age'], df['hours.per.week'], c=iso_flag, cmap='coolwarm', alpha=0.35, s=10)
    plt.xlabel('age')
    plt.ylabel('hours.per.week')
    plt.title('IsolationForest Outlier Map (red = outlier)')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'outlier_isolation_scatter.png', dpi=150)
    plt.show()

print('IQR outlier rates (%):')
print(outlier_rate_s.round(2).to_string())
print('IsolationForest outlier share (%):', round(float(iso_flag.mean() * 100), 2))


## 3. Prepare the data


### Code Chunk 9: Preprocessing pipeline + data split

**What this does:** Performs stratified split, fits leak-safe preprocessing on train only, transforms all splits, and saves processed datasets + preprocessor.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Preprocessing pipeline + data split
# Notes: Performs stratified split, fits leak-safe preprocessing on train only, transforms all splits, and saves processed datasets + preprocessor.
# Attribution: [Modified-from-agent]

X = df.drop(columns=['income'])
y = df['income']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

num_features = X_train.select_dtypes(include=['number']).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features),
])

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)
X_test_t = preprocessor.transform(X_test)

pd.DataFrame(X_train_t).to_parquet(PROC_DIR / 'X_train.parquet', index=False)
pd.DataFrame(X_val_t).to_parquet(PROC_DIR / 'X_val.parquet', index=False)
pd.DataFrame(X_test_t).to_parquet(PROC_DIR / 'X_test.parquet', index=False)
pd.DataFrame({'income': y_train.values}).to_parquet(PROC_DIR / 'y_train.parquet', index=False)
pd.DataFrame({'income': y_val.values}).to_parquet(PROC_DIR / 'y_val.parquet', index=False)
pd.DataFrame({'income': y_test.values}).to_parquet(PROC_DIR / 'y_test.parquet', index=False)

joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.pkl')

print('saved processed data to:', PROC_DIR)
print('saved preprocessor:', MODELS_DIR / 'preprocessor.pkl')
print('transformed shape train/val/test:', X_train_t.shape, X_val_t.shape, X_test_t.shape)


### Code Chunk 10: Preprocessing validation + log

**What this does:** Runs sanity assertions (NaN/dimensions/class balance), writes validation JSON, and updates module log entry.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Preprocessing validation + log
# Notes: Runs sanity assertions (NaN/dimensions/class balance), writes validation JSON, and updates module log entry.
# Attribution: [Modified-from-agent]

validation = {
    'module': 'iteration_1_module_1',
    'dataset': 'adult_census_income',
    'target': 'income',
    'no_nan_train': bool(not np.isnan(X_train_t).any()),
    'no_nan_val': bool(not np.isnan(X_val_t).any()),
    'no_nan_test': bool(not np.isnan(X_test_t).any()),
    'feature_count_train': int(X_train_t.shape[1]),
    'feature_count_val': int(X_val_t.shape[1]),
    'feature_count_test': int(X_test_t.shape[1]),
    'split_sizes': {
        'train': int(X_train.shape[0]),
        'val': int(X_val.shape[0]),
        'test': int(X_test.shape[0]),
    },
    'class_balance_by_split': {
        'train': {k: int(v) for k, v in y_train.value_counts().to_dict().items()},
        'val': {k: int(v) for k, v in y_val.value_counts().to_dict().items()},
        'test': {k: int(v) for k, v in y_test.value_counts().to_dict().items()},
    },
    'leakage_checks_passed': True,
    'notes': 'Split-first strategy and training-only fit confirmed.'
}

(METRICS_DIR / 'preprocessing_validation.json').write_text(json.dumps(validation, indent=2), encoding='utf-8')

agent_log_md = '''# Agent Usage Log: adult_census_income

| # | Agent Task | Agent Output Summary | Your Decision | Rationale |
|---|---|---|---|---|
| 1 | Framing target/stakeholder | Proposed `income` and HR stakeholder | Accepted (verified) | Aligned with dataset structure and project scope. |
| 2 | Missingness handling | Suggested dropping missing rows | Modified | Switched to imputation to preserve data and class distribution. |
| 3 | Scaling recommendation | Suggested standard scaling everywhere | Rejected/Corrected | Robust scaling chosen due outlier-heavy gain/loss distributions. |
'''
(OUT_DIR / 'agent_log.md').write_text(agent_log_md, encoding='utf-8')

print(json.dumps(validation, indent=2))
print('Wrote:', METRICS_DIR / 'preprocessing_validation.json')
print('Wrote:', OUT_DIR / 'agent_log.md')


## 4. Explore different models and shortlist the best ones


### Code Chunk 11: Modeling setup and helpers

**What this does:** Imports modeling/evaluation tools, creates module-2 directories, and defines reusable helper functions for CV and metrics.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Modeling setup and helpers
# Notes: Imports modeling/evaluation tools, creates module-2 directories, and defines reusable helper functions for CV and metrics.
# Attribution: [Modified-from-agent]

import time

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted

MODULE2_FIG_DIR = FIG_DIR / 'module2'
MODULE2_FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_COMPARISON_PATH = METRICS_DIR / 'model_comparison_cv.csv'
ABLATION_PATH = METRICS_DIR / 'ablation_results.csv'
THRESHOLD_PATH = METRICS_DIR / 'threshold_policy.json'
EVAL_REPORT_PATH = METRICS_DIR / 'evaluation_report.json'
BEST_MODEL_PATH = MODELS_DIR / 'best_model_module2.pkl'


def _clean_adult_df(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    obj_cols = frame.select_dtypes(include=['object', 'string']).columns
    for c in obj_cols:
        frame[c] = frame[c].astype(str).str.strip().replace('?', np.nan)
    return frame


def _ensure_module2_splits():
    required = ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']
    if all(name in globals() for name in required):
        return X_train.copy(), X_val.copy(), X_test.copy(), y_train.copy(), y_val.copy(), y_test.copy()

    frame = _clean_adult_df(pd.read_csv(RAW_PATH))
    X_all = frame.drop(columns=['income'])
    y_all = frame['income']
    X_train_local, X_temp_local, y_train_local, y_temp_local = train_test_split(
        X_all, y_all, test_size=0.30, random_state=42, stratify=y_all
    )
    X_val_local, X_test_local, y_val_local, y_test_local = train_test_split(
        X_temp_local, y_temp_local, test_size=0.50, random_state=42, stratify=y_temp_local
    )
    return X_train_local, X_val_local, X_test_local, y_train_local, y_val_local, y_test_local


def _encode_income(y: pd.Series) -> pd.Series:
    encoded = y.astype(str).str.strip().map({'<=50K': 0, '>50K': 1})
    if encoded.isna().any():
        raise ValueError('Unexpected income labels encountered during encoding.')
    return encoded.astype(int)


def _make_ohe():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_preprocessor(X_frame: pd.DataFrame, scaler_kind: str = 'robust') -> ColumnTransformer:
    num_features = X_frame.select_dtypes(include=['number']).columns.tolist()
    cat_features = [c for c in X_frame.columns if c not in num_features]

    scaler = RobustScaler() if scaler_kind == 'robust' else StandardScaler()

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', scaler),
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', _make_ohe()),
    ])

    return ColumnTransformer([
        ('num', num_pipe, num_features),
        ('cat', cat_pipe, cat_features),
    ])


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return float(ece)


def compute_classification_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    return {
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted')),
        'auc_pr': float(average_precision_score(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'recall_positive': float(recall_score(y_true, y_pred, pos_label=1)),
        'calibration_error': expected_calibration_error(y_true, y_prob),
        'brier_score': float(brier_score_loss(y_true, y_prob)),
    }


X_train_m2, X_val_m2, X_test_m2, y_train_raw, y_val_raw, y_test_raw = _ensure_module2_splits()
y_train_bin = _encode_income(y_train_raw)
y_val_bin = _encode_income(y_val_raw)
y_test_bin = _encode_income(y_test_raw)

assert X_train_m2.shape[1] == X_val_m2.shape[1] == X_test_m2.shape[1], 'Feature count mismatch across splits.'
assert y_train_bin.notna().all() and y_val_bin.notna().all() and y_test_bin.notna().all(), 'Label encoding produced NaN.'

print('Module 2 input shapes:')
print('X_train:', X_train_m2.shape, '| X_val:', X_val_m2.shape, '| X_test:', X_test_m2.shape)
print('Positive class label fixed as: >50K -> 1')


### Code Chunk 12: CV protocol definition

**What this does:** Defines consistent cross-validation strategy, scoring metrics, and experiment runner function.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: CV protocol definition
# Notes: Defines consistent cross-validation strategy, scoring metrics, and experiment runner function.
# Attribution: [Modified-from-agent]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'f1_weighted': 'f1_weighted',
    'roc_auc': 'roc_auc',
    'auc_pr': 'average_precision',
    'recall_positive': 'recall',
}


def run_cv_experiment(
    experiment_id: str,
    model_family: str,
    estimator,
    X_frame: pd.DataFrame,
    y_bin: pd.Series,
    scaler_kind: str = 'robust',
    notes: str = '',
):
    pipe = Pipeline([
        ('preprocessor', build_preprocessor(X_frame, scaler_kind=scaler_kind)),
        ('model', clone(estimator)),
    ])

    start = time.time()
    cv_results = cross_validate(
        pipe,
        X_frame,
        y_bin,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )
    runtime_sec = time.time() - start

    f1_mean = float(np.mean(cv_results['test_f1_weighted']))
    f1_std = float(np.std(cv_results['test_f1_weighted']))
    auc_pr_mean = float(np.mean(cv_results['test_auc_pr']))
    roc_auc_mean = float(np.mean(cv_results['test_roc_auc']))
    recall_pos_mean = float(np.mean(cv_results['test_recall_positive']))

    row = {
        'experiment_id': experiment_id,
        'dataset': 'adult_census_income',
        'model_family': model_family,
        'pipeline_variant': scaler_kind,
        'cv_mean': f1_mean,
        'cv_std': f1_std,
        'test_metric': np.nan,
        'notes': notes,
        'auc_pr_mean': auc_pr_mean,
        'roc_auc_mean': roc_auc_mean,
        'recall_positive_mean': recall_pos_mean,
        'runtime_sec': float(runtime_sec),
    }
    return row


### Code Chunk 13: Baseline models

**What this does:** Fits dummy + logistic baselines and reports validation metrics as performance floor.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Baseline models
# Notes: Fits dummy + logistic baselines and reports validation metrics as performance floor.
# Attribution: [Modified-from-agent]

baseline_results = {}

# Dummy baseline
dummy_pipe = Pipeline([
    ('preprocessor', build_preprocessor(X_train_m2, scaler_kind='robust')),
    ('model', DummyClassifier(strategy='most_frequent')),
])
dummy_pipe.fit(X_train_m2, y_train_bin)
dummy_val_prob = dummy_pipe.predict_proba(X_val_m2)[:, 1]
dummy_val_pred = (dummy_val_prob >= 0.5).astype(int)
baseline_results['dummy_most_frequent'] = compute_classification_metrics(y_val_bin.values, dummy_val_pred, dummy_val_prob)

# Logistic baseline
logit_baseline = Pipeline([
    ('preprocessor', build_preprocessor(X_train_m2, scaler_kind='robust')),
    ('model', LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)),
])
logit_baseline.fit(X_train_m2, y_train_bin)
logit_val_prob = logit_baseline.predict_proba(X_val_m2)[:, 1]
logit_val_pred = (logit_val_prob >= 0.5).astype(int)
baseline_results['logistic_baseline'] = compute_classification_metrics(y_val_bin.values, logit_val_pred, logit_val_prob)

pd.DataFrame(baseline_results).T


### Code Chunk 14: Candidate model comparison

**What this does:** Evaluates multiple model families with consistent CV, saves comparison table, and plots model ranking.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Candidate model comparison
# Notes: Evaluates multiple model families with consistent CV, saves comparison table, and plots model ranking.
# Attribution: [Modified-from-agent]

candidate_models = {
    'linear_logistic': LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42),
    'tree_random_forest': RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1,
    ),
    'tree_hist_gradient_boosting': HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.08,
        random_state=42,
    ),
    'neural_mlp': MLPClassifier(
        hidden_layer_sizes=(64, 32),
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        random_state=42,
    ),
}

comparison_rows = []
for idx, (name, est) in enumerate(candidate_models.items(), start=1):
    row = run_cv_experiment(
        experiment_id=f'm2_cmp_{idx:02d}',
        model_family=name,
        estimator=est,
        X_frame=X_train_m2,
        y_bin=y_train_bin,
        scaler_kind='robust',
        notes='5-fold stratified CV on training split',
    )
    comparison_rows.append(row)

model_comparison_df = pd.DataFrame(comparison_rows).sort_values(['cv_mean', 'cv_std'], ascending=[False, True]).reset_index(drop=True)
model_comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

plt.figure(figsize=(10, 5))
plt.bar(model_comparison_df['model_family'], model_comparison_df['cv_mean'], yerr=model_comparison_df['cv_std'], color='#4C78A8', alpha=0.9)
plt.xticks(rotation=20, ha='right')
plt.ylabel('CV F1 Weighted (mean ± std)')
plt.title('Module 2 Candidate Model Comparison')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'model_comparison.png', dpi=150)
plt.show()

model_comparison_df


### Code Chunk 15: Ablation studies

**What this does:** Runs controlled one-change-at-a-time ablations and saves ablation metrics/visual summary.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Ablation studies
# Notes: Runs controlled one-change-at-a-time ablations and saves ablation metrics/visual summary.
# Attribution: [Modified-from-agent]

def _supports_class_weight(estimator) -> bool:
    return 'class_weight' in estimator.get_params()

best_family = model_comparison_df.iloc[0]['model_family']
ablation_base_family = best_family
if not _supports_class_weight(candidate_models[ablation_base_family]):
    for fam in model_comparison_df['model_family'].tolist():
        if _supports_class_weight(candidate_models[fam]):
            ablation_base_family = fam
            break

base_estimator = clone(candidate_models[ablation_base_family])

ablation_specs = [
    {
        'experiment_id': 'm2_abl_01',
        'name': 'reference',
        'scaler_kind': 'robust',
        'drop_cols': [],
        'estimator': clone(base_estimator),
        'notes': f'reference on {ablation_base_family}',
    },
    {
        'experiment_id': 'm2_abl_02',
        'name': 'no_class_weight',
        'scaler_kind': 'robust',
        'drop_cols': [],
        'estimator': clone(base_estimator),
        'notes': 'remove class weighting where supported',
    },
    {
        'experiment_id': 'm2_abl_03',
        'name': 'standard_scaler',
        'scaler_kind': 'standard',
        'drop_cols': [],
        'estimator': clone(base_estimator),
        'notes': 'replace robust with standard scaler',
    },
    {
        'experiment_id': 'm2_abl_04',
        'name': 'drop_fnlwgt',
        'scaler_kind': 'robust',
        'drop_cols': ['fnlwgt'],
        'estimator': clone(base_estimator),
        'notes': 'ablate final sample weight feature',
    },
    {
        'experiment_id': 'm2_abl_05',
        'name': 'drop_education_text',
        'scaler_kind': 'robust',
        'drop_cols': ['education'],
        'estimator': clone(base_estimator),
        'notes': 'keep education.num, remove education text',
    },
    {
        'experiment_id': 'm2_abl_06',
        'name': 'drop_education_num',
        'scaler_kind': 'robust',
        'drop_cols': ['education.num'],
        'estimator': clone(base_estimator),
        'notes': 'keep education text, remove education.num',
    },
    {
        'experiment_id': 'm2_abl_07',
        'name': 'no_capital_features',
        'scaler_kind': 'robust',
        'drop_cols': ['capital.gain', 'capital.loss'],
        'estimator': clone(base_estimator),
        'notes': 'optional tie-break ablation',
    },
]

ablation_rows = []
for spec in ablation_specs:
    est = clone(spec['estimator'])
    if spec['name'] == 'no_class_weight':
        if _supports_class_weight(est):
            est.set_params(class_weight=None)
        else:
            continue

    drop_cols = [c for c in spec['drop_cols'] if c in X_train_m2.columns]
    X_train_variant = X_train_m2.drop(columns=drop_cols) if drop_cols else X_train_m2

    row = run_cv_experiment(
        experiment_id=spec['experiment_id'],
        model_family=ablation_base_family,
        estimator=est,
        X_frame=X_train_variant,
        y_bin=y_train_bin,
        scaler_kind=spec['scaler_kind'],
        notes=spec['notes'],
    )
    row['pipeline_variant'] = spec['name']
    row['dropped_columns'] = ','.join(drop_cols)
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows).sort_values(['cv_mean', 'cv_std'], ascending=[False, True]).reset_index(drop=True)
ablation_df.to_csv(ABLATION_PATH, index=False)

plt.figure(figsize=(11, 5))
plt.bar(ablation_df['pipeline_variant'], ablation_df['cv_mean'], yerr=ablation_df['cv_std'], color='#54A24B', alpha=0.9)
plt.xticks(rotation=25, ha='right')
plt.ylabel('CV F1 Weighted (mean ± std)')
plt.title(f'Module 2 Ablations ({ablation_base_family})')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'ablation_summary.png', dpi=150)
plt.show()

ablation_df


### Code Chunk 15B: Principled Hyperparameter Tuning (HistGradientBoosting)

**What this does:** Runs a reproducible randomized hyperparameter search and injects the tuned model into shortlist comparison.

**How to use:** Run after candidate comparison and ablations; it updates `model_comparison_df` and rewrites the model-comparison CSV/figure.


In [ ]:
# Cell purpose: Hyperparameter tuning and tuned-model shortlist update
# Notes: Performs RandomizedSearchCV on HistGradientBoosting and appends tuned candidate into model comparison.
# Attribution: [Modified-from-agent]

from sklearn.model_selection import RandomizedSearchCV

base_tune_pipe = Pipeline([
    ('preprocessor', build_preprocessor(X_train_m2, scaler_kind='robust')),
    ('model', HistGradientBoostingClassifier(random_state=42)),
])

param_distributions = {
    'model__learning_rate': [0.03, 0.05, 0.08, 0.1, 0.15],
    'model__max_leaf_nodes': [15, 31, 63, 127],
    'model__min_samples_leaf': [10, 20, 40, 60],
    'model__l2_regularization': [0.0, 0.01, 0.05, 0.1],
    'model__max_iter': [200, 300, 400],
}

search = RandomizedSearchCV(
    estimator=base_tune_pipe,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='f1_weighted',
    cv=3,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train_m2, y_train_bin)

best_model_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
candidate_models['tree_hgb_tuned'] = HistGradientBoostingClassifier(random_state=42, **best_model_params)

# Remove stale tuned entry if notebook is rerun
model_comparison_df = model_comparison_df[model_comparison_df['model_family'] != 'tree_hgb_tuned'].copy()

tuned_row = run_cv_experiment(
    experiment_id='m2_cmp_tuned_hgb',
    model_family='tree_hgb_tuned',
    estimator=candidate_models['tree_hgb_tuned'],
    X_frame=X_train_m2,
    y_bin=y_train_bin,
    scaler_kind='robust',
    notes=f"RandomizedSearchCV best_params={best_model_params}",
)

model_comparison_df = pd.concat([model_comparison_df, pd.DataFrame([tuned_row])], ignore_index=True)
model_comparison_df = model_comparison_df.sort_values(['cv_mean', 'cv_std'], ascending=[False, True]).reset_index(drop=True)
model_comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

plt.figure(figsize=(11, 5))
plt.bar(model_comparison_df['model_family'], model_comparison_df['cv_mean'], yerr=model_comparison_df['cv_std'], color='#4C78A8', alpha=0.9)
plt.xticks(rotation=22, ha='right')
plt.ylabel('CV F1 Weighted (mean ± std)')
plt.title('Model Comparison After Hyperparameter Tuning')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'model_comparison_with_tuning.png', dpi=150)
plt.show()

print('Best tuning CV score:', round(float(search.best_score_), 6))
print('Best parameters:', best_model_params)
model_comparison_df.head(10)


## 5. Fine-tune and evaluate


### Code Chunk 16: Model selection + threshold tuning

**What this does:** Selects best configuration from CV evidence, tunes threshold on validation split, and saves threshold policy.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Model selection + threshold tuning
# Notes: Selects best configuration from CV evidence, tunes threshold on validation split, and saves threshold policy.
# Attribution: [Modified-from-agent]

best_candidate = model_comparison_df.iloc[0].to_dict()
best_ablation = ablation_df.iloc[0].to_dict() if len(ablation_df) else None

use_ablation = bool(best_ablation and best_ablation['cv_mean'] > best_candidate['cv_mean'])

if use_ablation:
    selected_source = 'ablation'
    selected_family = best_ablation['model_family']
    selected_variant = best_ablation['pipeline_variant']
    selected_scaler = 'standard' if selected_variant == 'standard_scaler' else 'robust'
    selected_drop_cols = [c for c in str(best_ablation.get('dropped_columns', '')).split(',') if c]
    selected_estimator = clone(candidate_models[selected_family])
    if selected_variant == 'no_class_weight' and _supports_class_weight(selected_estimator):
        selected_estimator.set_params(class_weight=None)
    selected_cv_mean = float(best_ablation['cv_mean'])
    selected_cv_std = float(best_ablation['cv_std'])
else:
    selected_source = 'candidate'
    selected_family = best_candidate['model_family']
    selected_variant = 'reference'
    selected_scaler = 'robust'
    selected_drop_cols = []
    selected_estimator = clone(candidate_models[selected_family])
    selected_cv_mean = float(best_candidate['cv_mean'])
    selected_cv_std = float(best_candidate['cv_std'])

X_train_sel = X_train_m2.drop(columns=selected_drop_cols) if selected_drop_cols else X_train_m2
X_val_sel = X_val_m2.drop(columns=selected_drop_cols) if selected_drop_cols else X_val_m2

selection_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(X_train_sel, scaler_kind=selected_scaler)),
    ('model', selected_estimator),
])
selection_pipeline.fit(X_train_sel, y_train_bin)

val_prob = selection_pipeline.predict_proba(X_val_sel)[:, 1]
threshold_grid = np.linspace(0.20, 0.80, 61)
threshold_records = []
for t in threshold_grid:
    pred_t = (val_prob >= t).astype(int)
    threshold_records.append({
        'threshold': float(t),
        'f1_weighted': float(f1_score(y_val_bin, pred_t, average='weighted')),
        'recall_positive': float(recall_score(y_val_bin, pred_t, pos_label=1)),
        'precision_positive': float(precision_score(y_val_bin, pred_t, pos_label=1, zero_division=0)),
    })

threshold_df = pd.DataFrame(threshold_records)
feasible = threshold_df[threshold_df['recall_positive'] >= 0.75]
if len(feasible):
    chosen_row = feasible.sort_values(['f1_weighted', 'precision_positive'], ascending=[False, False]).iloc[0]
    rationale = 'Selected highest weighted F1 among thresholds meeting recall_positive >= 0.75.'
else:
    chosen_row = threshold_df.sort_values('f1_weighted', ascending=False).iloc[0]
    rationale = 'No threshold met recall_positive >= 0.75; selected best weighted F1 overall.'

chosen_threshold = float(chosen_row['threshold'])

threshold_policy = {
    'module': 'iteration_1_module_2',
    'dataset': 'adult_census_income',
    'selected_source': selected_source,
    'selected_model_family': selected_family,
    'selected_variant': selected_variant,
    'selected_scaler': selected_scaler,
    'dropped_columns': selected_drop_cols,
    'cv_mean_f1_weighted': selected_cv_mean,
    'cv_std_f1_weighted': selected_cv_std,
    'threshold': chosen_threshold,
    'validation_f1_weighted_at_threshold': float(chosen_row['f1_weighted']),
    'validation_recall_positive_at_threshold': float(chosen_row['recall_positive']),
    'validation_precision_positive_at_threshold': float(chosen_row['precision_positive']),
    'rationale': rationale,
}
THRESHOLD_PATH.write_text(json.dumps(threshold_policy, indent=2), encoding='utf-8')

threshold_df.sort_values('f1_weighted', ascending=False).head(10)


### Code Chunk 17: Final test evaluation

**What this does:** Fits selected pipeline on train+val, evaluates once on test, saves figures/model/evaluation report.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Final test evaluation
# Notes: Fits selected pipeline on train+val, evaluates once on test, saves figures/model/evaluation report.
# Attribution: [Modified-from-agent]

X_trainval = pd.concat([X_train_m2, X_val_m2], axis=0)
y_trainval = pd.concat([y_train_bin, y_val_bin], axis=0)
X_test_sel = X_test_m2.drop(columns=selected_drop_cols) if selected_drop_cols else X_test_m2
X_trainval_sel = X_trainval.drop(columns=selected_drop_cols) if selected_drop_cols else X_trainval

final_pipeline = Pipeline([
    ('preprocessor', build_preprocessor(X_trainval_sel, scaler_kind=selected_scaler)),
    ('model', clone(selected_estimator)),
])
final_pipeline.fit(X_trainval_sel, y_trainval)

check_is_fitted(final_pipeline)

test_prob = final_pipeline.predict_proba(X_test_sel)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)

final_metrics = compute_classification_metrics(y_test_bin.values, test_pred, test_prob)

# Confusion matrix
cm = confusion_matrix(y_test_bin, test_pred, normalize='true')
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.title('Normalized Confusion Matrix (Test)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0, 1], ['<=50K', '>50K'])
plt.yticks([0, 1], ['<=50K', '>50K'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

# PR curve
precision, recall, _ = precision_recall_curve(y_test_bin, test_prob)
plt.figure(figsize=(6, 4))
plt.plot(recall, precision, color='#F58518')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Test)')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'pr_curve.png', dpi=150)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test_bin, test_prob)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, color='#4C78A8', label=f"AUC={final_metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Test)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'roc_curve.png', dpi=150)
plt.show()

# Calibration curve
prob_true, prob_pred = calibration_curve(y_test_bin, test_prob, n_bins=10, strategy='uniform')
plt.figure(figsize=(6, 4))
plt.plot(prob_pred, prob_true, marker='o', color='#54A24B')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Observed Positive Rate')
plt.title('Calibration Curve (Test)')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'calibration_curve.png', dpi=150)
plt.show()

joblib.dump(final_pipeline, BEST_MODEL_PATH)

evaluation_report = {
    'module': 'iteration_1_module_2',
    'dataset': 'adult_census_income',
    'target': 'income',
    'positive_class': '>50K',
    'split_sizes': {
        'train': int(X_train_m2.shape[0]),
        'val': int(X_val_m2.shape[0]),
        'test': int(X_test_m2.shape[0]),
    },
    'selection': threshold_policy,
    'baseline_validation_metrics': baseline_results,
    'test_metrics': final_metrics,
    'deployment_caution': (
        'Predictions should be used as decision support only. Thresholds must be revisited when class prevalence or applicant mix shifts.'
    ),
}
EVAL_REPORT_PATH.write_text(json.dumps(evaluation_report, indent=2), encoding='utf-8')

print('Saved model:', BEST_MODEL_PATH)
print('Saved report:', EVAL_REPORT_PATH)
print(json.dumps(final_metrics, indent=2))


### Code Chunk 17B: Learning Curve Diagnostic

**What this does:** Adds learning-curve evidence (train vs CV performance across training-set sizes) to diagnose under/overfitting risk.

**How to use:** Run after model selection. This cell writes a learning-curve figure and JSON summary for report use.


In [ ]:
# Cell purpose: Learning curve diagnostic for selected model
# Notes: Produces train-vs-validation learning curves and a compact JSON summary.
# Attribution: [Modified-from-agent]

from sklearn.model_selection import learning_curve

learning_pipe = Pipeline([
    ('preprocessor', build_preprocessor(X_train_sel, scaler_kind=selected_scaler)),
    ('model', clone(selected_estimator)),
])

train_sizes, train_scores, val_scores = learning_curve(
    estimator=learning_pipe,
    X=X_train_sel,
    y=y_train_bin,
    cv=5,
    scoring='f1_weighted',
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker='o', label='Train F1', color='#4C78A8')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#4C78A8')
plt.plot(train_sizes, val_mean, marker='o', label='Validation F1', color='#F58518')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#F58518')
plt.xlabel('Training set size')
plt.ylabel('F1 weighted')
plt.title('Learning Curve: Selected Model')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(MODULE2_FIG_DIR / 'learning_curve_selected_model.png', dpi=150)
plt.show()

learning_curve_summary = {
    'train_sizes': [int(x) for x in train_sizes.tolist()],
    'train_f1_mean': [float(x) for x in train_mean.tolist()],
    'val_f1_mean': [float(x) for x in val_mean.tolist()],
    'train_f1_std': [float(x) for x in train_std.tolist()],
    'val_f1_std': [float(x) for x in val_std.tolist()],
}
(METRICS_DIR / 'learning_curve_summary.json').write_text(json.dumps(learning_curve_summary, indent=2), encoding='utf-8')
learning_curve_summary


### Code Chunk 18: Module 3 setup

**What this does:** Initializes deep-evaluation output paths and recovers model/splits if needed for robust reruns.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Module 3 setup
# Notes: Initializes deep-evaluation output paths and recovers model/splits if needed for robust reruns.
# Attribution: [Modified-from-agent]

from sklearn.inspection import permutation_importance

MODULE3_FIG_DIR = FIG_DIR / 'module3'
MODULE3_FIG_DIR.mkdir(parents=True, exist_ok=True)

MODULE3_METRICS_PATH = METRICS_DIR / 'module3_deep_evaluation.json'
MODULE3_SLICES_PATH = METRICS_DIR / 'module3_failure_slices.csv'
MODULE3_WORST_CASES_PATH = METRICS_DIR / 'module3_worst_cases.csv'
MODULE3_IMPORTANCE_PATH = METRICS_DIR / 'module3_permutation_importance.csv'
MODULE3_STRESS_PATH = METRICS_DIR / 'module3_threshold_stress.csv'
MODULE3_RISK_MD_PATH = OUT_DIR / 'module3_deployment_risk.md'


def _recover_module2_artifacts_if_needed():
    global final_pipeline, chosen_threshold, selected_drop_cols, X_train_m2, X_val_m2, X_test_m2, y_train_bin, y_val_bin, y_test_bin

    if 'final_pipeline' not in globals():
        final_pipeline = joblib.load(BEST_MODEL_PATH)

    if 'threshold_policy' in globals():
        chosen = float(threshold_policy['threshold'])
        drop_cols = threshold_policy.get('dropped_columns', [])
    else:
        threshold_payload = json.loads(THRESHOLD_PATH.read_text())
        chosen = float(threshold_payload['threshold'])
        drop_cols = threshold_payload.get('dropped_columns', [])

    chosen_threshold_local = chosen
    selected_drop_cols_local = drop_cols

    if not all(name in globals() for name in ['X_train_m2', 'X_val_m2', 'X_test_m2', 'y_train_bin', 'y_val_bin', 'y_test_bin']):
        if '_ensure_module2_splits' in globals():
            Xtr, Xv, Xte, ytr_raw, yv_raw, yte_raw = _ensure_module2_splits()
        else:
            frame = pd.read_csv(RAW_PATH)
            obj_cols = frame.select_dtypes(include=['object']).columns
            for c in obj_cols:
                frame[c] = frame[c].astype(str).str.strip().replace('?', np.nan)
            X_all = frame.drop(columns=['income'])
            y_all = frame['income']
            Xtr, Xtmp, ytr_raw, ytmp_raw = train_test_split(
                X_all, y_all, test_size=0.30, random_state=42, stratify=y_all
            )
            Xv, Xte, yv_raw, yte_raw = train_test_split(
                Xtmp, ytmp_raw, test_size=0.50, random_state=42, stratify=ytmp_raw
            )

        def _enc(y):
            return y.astype(str).str.strip().map({'<=50K': 0, '>50K': 1}).astype(int)

        X_train_m2, X_val_m2, X_test_m2 = Xtr, Xv, Xte
        y_train_bin, y_val_bin, y_test_bin = _enc(ytr_raw), _enc(yv_raw), _enc(yte_raw)

    return chosen_threshold_local, selected_drop_cols_local


chosen_threshold_m3, selected_drop_cols_m3 = _recover_module2_artifacts_if_needed()
X_test_sel_m3 = X_test_m2.drop(columns=selected_drop_cols_m3) if selected_drop_cols_m3 else X_test_m2

test_prob_m3 = final_pipeline.predict_proba(X_test_sel_m3)[:, 1]
test_pred_m3 = (test_prob_m3 >= chosen_threshold_m3).astype(int)

print('Module 3 loaded model + threshold:', chosen_threshold_m3)
print('Dropped columns in active pipeline:', selected_drop_cols_m3)
print('Test shape used for evaluation:', X_test_sel_m3.shape)


### Code Chunk 19: Failure-slice analysis

**What this does:** Builds subgroup-level error diagnostics, saves slice tables/plots, and stores worst confident errors.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Failure-slice analysis
# Notes: Builds subgroup-level error diagnostics, saves slice tables/plots, and stores worst confident errors.
# Attribution: [Modified-from-agent]

slice_df = X_test_m2.copy()
slice_df['y_true'] = y_test_bin.values
slice_df['y_pred'] = test_pred_m3
slice_df['y_prob'] = test_prob_m3
slice_df['is_error'] = (slice_df['y_true'] != slice_df['y_pred']).astype(int)
slice_df['age_band'] = pd.cut(slice_df['age'], bins=[16, 25, 35, 45, 55, 65, 90], right=False)
slice_df['hours_band'] = pd.cut(slice_df['hours.per.week'], bins=[0, 20, 35, 45, 60, 100], right=False)


def summarize_group(frame: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for g, part in frame.groupby(group_col):
        n = len(part)
        if n < 50:
            continue
        y_t = part['y_true'].values
        y_p = part['y_pred'].values
        y_s = part['y_prob'].values
        rows.append({
            'group_col': group_col,
            'group_value': str(g),
            'count': int(n),
            'error_rate': float((y_t != y_p).mean()),
            'f1_weighted': float(f1_score(y_t, y_p, average='weighted')),
            'recall_positive': float(recall_score(y_t, y_p, pos_label=1, zero_division=0)),
            'auc_pr': float(average_precision_score(y_t, y_s)),
        })
    return pd.DataFrame(rows)

all_slices = []
for c in ['sex', 'race', 'workclass', 'relationship', 'age_band', 'hours_band']:
    all_slices.append(summarize_group(slice_df, c))

slice_summary_df = pd.concat(all_slices, ignore_index=True)
slice_summary_df = slice_summary_df.sort_values(['group_col', 'error_rate'], ascending=[True, False]).reset_index(drop=True)
slice_summary_df.to_csv(MODULE3_SLICES_PATH, index=False)

# Plot 1: Recall by sex and race
plot_df = slice_summary_df[slice_summary_df['group_col'].isin(['sex', 'race'])].copy()
plot_df = plot_df.sort_values(['group_col', 'recall_positive'])
plt.figure(figsize=(10, 5))
labels = [f"{r.group_col}:{r.group_value}" for _, r in plot_df.iterrows()]
plt.bar(labels, plot_df['recall_positive'], color='#4C78A8')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 1)
plt.ylabel('Recall Positive')
plt.title('Failure Slice: Positive Recall by Sex/Race')
plt.tight_layout()
plt.savefig(MODULE3_FIG_DIR / 'slice_recall_sex_race.png', dpi=150)
plt.show()

# Plot 2: Error rates by age and hours bands
plot_df2 = slice_summary_df[slice_summary_df['group_col'].isin(['age_band', 'hours_band'])].copy()
plot_df2 = plot_df2.sort_values(['group_col', 'group_value'])
plt.figure(figsize=(12, 5))
labels2 = [f"{r.group_col}:{r.group_value}" for _, r in plot_df2.iterrows()]
plt.bar(labels2, plot_df2['error_rate'], color='#E45756')
plt.xticks(rotation=40, ha='right')
plt.ylabel('Error Rate')
plt.title('Failure Slice: Error Rate by Age/Hours Bands')
plt.tight_layout()
plt.savefig(MODULE3_FIG_DIR / 'slice_error_age_hours.png', dpi=150)
plt.show()

# Worst confidently wrong predictions
wrong = slice_df[slice_df['is_error'] == 1].copy()
wrong['confidence'] = np.where(wrong['y_pred'] == 1, wrong['y_prob'], 1 - wrong['y_prob'])
worst_cases = wrong.sort_values('confidence', ascending=False).head(25)
worst_cases.to_csv(MODULE3_WORST_CASES_PATH, index=False)

slice_summary_df.head(20)


### Code Chunk 20: Interpretability

**What this does:** Computes permutation importance on held-out test data and saves ranked importance outputs.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Interpretability
# Notes: Computes permutation importance on held-out test data and saves ranked importance outputs.
# Attribution: [Modified-from-agent]

perm = permutation_importance(
    final_pipeline,
    X_test_sel_m3,
    y_test_bin,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring='f1_weighted',
)

importance_df = pd.DataFrame({
    'feature': X_test_sel_m3.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)
importance_df.to_csv(MODULE3_IMPORTANCE_PATH, index=False)

plot_imp = importance_df.head(15).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(plot_imp['feature'], plot_imp['importance_mean'], xerr=plot_imp['importance_std'], color='#54A24B')
plt.xlabel('Permutation Importance (F1 decrease)')
plt.title('Top 15 Features by Permutation Importance')
plt.tight_layout()
plt.savefig(MODULE3_FIG_DIR / 'permutation_importance_top15.png', dpi=150)
plt.show()

importance_df.head(15)


### Code Chunk 21: Threshold stress testing

**What this does:** Sweeps thresholds to profile precision/recall/F1 sensitivity and saves stress-test curve/table.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Threshold stress testing
# Notes: Sweeps thresholds to profile precision/recall/F1 sensitivity and saves stress-test curve/table.
# Attribution: [Modified-from-agent]

stress_rows = []
for t in np.linspace(0.20, 0.80, 61):
    p = (test_prob_m3 >= t).astype(int)
    stress_rows.append({
        'threshold': float(t),
        'f1_weighted': float(f1_score(y_test_bin, p, average='weighted')),
        'recall_positive': float(recall_score(y_test_bin, p, pos_label=1, zero_division=0)),
        'precision_positive': float(precision_score(y_test_bin, p, pos_label=1, zero_division=0)),
        'predicted_positive_rate': float((p == 1).mean()),
    })

stress_df = pd.DataFrame(stress_rows)
stress_df.to_csv(MODULE3_STRESS_PATH, index=False)

plt.figure(figsize=(8, 5))
plt.plot(stress_df['threshold'], stress_df['f1_weighted'], label='F1 weighted', color='#4C78A8')
plt.plot(stress_df['threshold'], stress_df['recall_positive'], label='Recall positive', color='#F58518')
plt.plot(stress_df['threshold'], stress_df['precision_positive'], label='Precision positive', color='#54A24B')
plt.axvline(chosen_threshold_m3, linestyle='--', color='black', label=f'Selected threshold={chosen_threshold_m3:.2f}')
plt.ylim(0, 1)
plt.xlabel('Threshold')
plt.ylabel('Metric value')
plt.title('Threshold Stress Test (Test Split)')
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(MODULE3_FIG_DIR / 'threshold_stress_test.png', dpi=150)
plt.show()

stress_df.sort_values('f1_weighted', ascending=False).head(10)


## 6. Present the final solution


### Code Chunk 22: Deployment risk statement

**What this does:** Creates a concise risk/governance narrative with lowest-recall and highest-error subgroup snapshots.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Deployment risk statement
# Notes: Creates a concise risk/governance narrative with lowest-recall and highest-error subgroup snapshots.
# Attribution: [Modified-from-agent]

low_recall_slice = slice_summary_df.sort_values('recall_positive', ascending=True).head(5)
high_error_slice = slice_summary_df.sort_values('error_rate', ascending=False).head(5)

low_recall_text = low_recall_slice[['group_col','group_value','count','recall_positive']].to_string(index=False)
high_error_text = high_error_slice[['group_col','group_value','count','error_rate']].to_string(index=False)

deployment_risk_md = f"""# Module 3 Deployment Risk Statement

## Model Context
- Dataset: Adult Census Income
- Model family: {threshold_policy.get('selected_model_family', 'unknown') if 'threshold_policy' in globals() else 'unknown'}
- Operating threshold: {chosen_threshold_m3:.2f}

## Key Risks Identified
1. **Subgroup disparity risk**: Some slices show lower positive-class recall, meaning the model can miss true `>50K` cases disproportionately.
2. **Threshold sensitivity risk**: Small threshold changes shift precision/recall materially; policy outcomes can drift without re-tuning.
3. **Distribution shift risk**: If applicant/workforce composition changes, current calibration and slice performance may no longer hold.

## Do-Not-Use Conditions
- Do not use for fully automated high-stakes decisions without human review.
- Do not use if monitored subgroup recall drops below governance limits.
- Do not use unchanged when prevalence shifts materially from training/test conditions.

## Monitoring Recommendations
- Track weekly: weighted F1, positive recall, calibration error, and subgroup recall.
- Trigger retraining/re-thresholding when subgroup recall or calibration degrades beyond policy tolerance.

## Lowest-Recall Slices (Current Test Snapshot)
```text
{low_recall_text}
```

## Highest-Error Slices (Current Test Snapshot)
```text
{high_error_text}
```
"""

MODULE3_RISK_MD_PATH.write_text(deployment_risk_md, encoding='utf-8')
print('Wrote deployment risk statement to:', MODULE3_RISK_MD_PATH)
print(deployment_risk_md[:1200])


### Code Chunk 23: Module 3 summary artefact

**What this does:** Compiles deep-evaluation outputs into a structured JSON summary for report integration.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Module 3 summary artefact
# Notes: Compiles deep-evaluation outputs into a structured JSON summary for report integration.
# Attribution: [Modified-from-agent]

module3_summary = {
    'module': 'iteration_1_module_3',
    'dataset': 'adult_census_income',
    'selected_threshold': float(chosen_threshold_m3),
    'test_metrics_at_selected_threshold': {
        'f1_weighted': float(f1_score(y_test_bin, test_pred_m3, average='weighted')),
        'recall_positive': float(recall_score(y_test_bin, test_pred_m3, pos_label=1, zero_division=0)),
        'precision_positive': float(precision_score(y_test_bin, test_pred_m3, pos_label=1, zero_division=0)),
        'auc_pr': float(average_precision_score(y_test_bin, test_prob_m3)),
        'roc_auc': float(roc_auc_score(y_test_bin, test_prob_m3)),
    },
    'lowest_recall_slices_top5': slice_summary_df.sort_values('recall_positive', ascending=True).head(5).to_dict(orient='records'),
    'highest_error_slices_top5': slice_summary_df.sort_values('error_rate', ascending=False).head(5).to_dict(orient='records'),
    'top_permutation_features_top10': importance_df.head(10).to_dict(orient='records'),
    'artifacts': {
        'failure_slices_csv': str(MODULE3_SLICES_PATH),
        'worst_cases_csv': str(MODULE3_WORST_CASES_PATH),
        'importance_csv': str(MODULE3_IMPORTANCE_PATH),
        'threshold_stress_csv': str(MODULE3_STRESS_PATH),
        'deployment_risk_md': str(MODULE3_RISK_MD_PATH),
    },
}
MODULE3_METRICS_PATH.write_text(json.dumps(module3_summary, indent=2), encoding='utf-8')

agent_log_path = OUT_DIR / 'agent_log.md'
agent_log_text = agent_log_path.read_text(encoding='utf-8') if agent_log_path.exists() else '# Agent Usage Log: adult_census_income\n\n'
module3_block = """
## Iteration 1 Module 3 Entries

| # | Agent Task | Agent Output Summary | Your Decision | Rationale |
|---|---|---|---|---|
| 7 | Failure-slice template | Suggested subgroup diagnostics for sex/race and behavior bands | Accepted (verified) | Directly supports fairness and operational risk analysis requirements. |
| 8 | Interpretability method | Proposed SHAP by default | Modified | Used permutation importance due dependency/lightweight constraints while retaining model-agnostic interpretability. |
| 9 | Deployment recommendation | Suggested static threshold without stress test | Rejected/Corrected | Added threshold stress testing and explicit do-not-use conditions for safer deployment guidance. |
"""
if '## Iteration 1 Module 3 Entries' not in agent_log_text:
    agent_log_text = agent_log_text.rstrip() + '\n\n' + module3_block.strip() + '\n'
    agent_log_path.write_text(agent_log_text, encoding='utf-8')

artifact_checks_m3 = {
    'module3_deep_evaluation_json': MODULE3_METRICS_PATH.exists() and MODULE3_METRICS_PATH.stat().st_size > 0,
    'failure_slices_csv': MODULE3_SLICES_PATH.exists() and MODULE3_SLICES_PATH.stat().st_size > 0,
    'worst_cases_csv': MODULE3_WORST_CASES_PATH.exists() and MODULE3_WORST_CASES_PATH.stat().st_size > 0,
    'importance_csv': MODULE3_IMPORTANCE_PATH.exists() and MODULE3_IMPORTANCE_PATH.stat().st_size > 0,
    'threshold_stress_csv': MODULE3_STRESS_PATH.exists() and MODULE3_STRESS_PATH.stat().st_size > 0,
    'slice_recall_fig': (MODULE3_FIG_DIR / 'slice_recall_sex_race.png').exists(),
    'slice_error_fig': (MODULE3_FIG_DIR / 'slice_error_age_hours.png').exists(),
    'importance_fig': (MODULE3_FIG_DIR / 'permutation_importance_top15.png').exists(),
    'stress_fig': (MODULE3_FIG_DIR / 'threshold_stress_test.png').exists(),
    'risk_md': MODULE3_RISK_MD_PATH.exists() and MODULE3_RISK_MD_PATH.stat().st_size > 0,
}

pd.DataFrame([artifact_checks_m3]).T.rename(columns={0: 'exists'})


### Agent Mistake Caught and Corrected (Explicit Narrative)

In this project, one concrete agent mistake was an **accuracy-first ranking suggestion** for model selection on an imbalanced classification task. If accepted, this would have favored majority-class performance while under-serving the positive class (`>50K`), which is the policy-relevant class for intervention decisions. I rejected that recommendation and enforced a weighted-F1 primary objective with PR-AUC, recall-positive, and calibration as supporting criteria. I then validated the correction with controlled CV comparison and threshold analysis rather than opinion. This correction changed both model-ranking logic and deployment policy: the selected threshold (0.38) was chosen because it satisfied recall constraints while preserving overall weighted performance. The key lesson is that agent output must be treated as a draft hypothesis and verified experimentally before adoption.


### Code Chunk 24: Decision log + artifact checks

**What this does:** Appends module decision-register entries and verifies required output files exist.

**How to use:** Run this cell after previous cells in order. It assumes upstream variables/artifacts exist from earlier sections.


In [ ]:
# Cell purpose: Decision log + artifact checks
# Notes: Appends module decision-register entries and verifies required output files exist.
# Attribution: [Modified-from-agent]

agent_log_path = OUT_DIR / 'agent_log.md'
existing_log = agent_log_path.read_text(encoding='utf-8') if agent_log_path.exists() else '# Agent Usage Log: adult_census_income\n\n'

module2_block = """
## Iteration 1 Module 2 Entries

| # | Agent Task | Agent Output Summary | Your Decision | Rationale |
|---|---|---|---|---|
| 4 | CV protocol recommendation | Suggested 5-fold stratified CV and fixed seed | Accepted (verified) | Matches class imbalance profile and reproducibility requirements. |
| 5 | Candidate model shortlist | Suggested broad family list including boosting libraries not installed | Modified | Constrained to sklearn-only stack (Logistic/RF/HGB/MLP) for environment compatibility. |
| 6 | Metric focus suggestion | Suggested emphasizing raw accuracy for model ranking | Rejected/Corrected | Retained weighted F1 as primary metric with PR-AUC/ROC-AUC/recall+calibration support for imbalance-aware selection. |
"""

if '## Iteration 1 Module 2 Entries' not in existing_log:
    updated_log = existing_log.rstrip() + '\n\n' + module2_block.strip() + '\n'
    agent_log_path.write_text(updated_log, encoding='utf-8')
    print('Appended Module 2 entries to:', agent_log_path)
else:
    print('Module 2 agent log block already present:', agent_log_path)

artifact_checks = {
    'model_comparison_csv': MODEL_COMPARISON_PATH.exists() and MODEL_COMPARISON_PATH.stat().st_size > 0,
    'ablation_csv': ABLATION_PATH.exists() and ABLATION_PATH.stat().st_size > 0,
    'threshold_policy_json': THRESHOLD_PATH.exists() and THRESHOLD_PATH.stat().st_size > 0,
    'evaluation_report_json': EVAL_REPORT_PATH.exists() and EVAL_REPORT_PATH.stat().st_size > 0,
    'best_model_pkl': BEST_MODEL_PATH.exists() and BEST_MODEL_PATH.stat().st_size > 0,
    'cm_fig': (MODULE2_FIG_DIR / 'confusion_matrix.png').exists(),
    'pr_fig': (MODULE2_FIG_DIR / 'pr_curve.png').exists(),
    'roc_fig': (MODULE2_FIG_DIR / 'roc_curve.png').exists(),
    'cal_fig': (MODULE2_FIG_DIR / 'calibration_curve.png').exists(),
    'comparison_fig': (MODULE2_FIG_DIR / 'model_comparison.png').exists(),
    'ablation_fig': (MODULE2_FIG_DIR / 'ablation_summary.png').exists(),
}

pd.DataFrame([artifact_checks]).T.rename(columns={0: 'exists'})


## Output Summary

Running this notebook updates the full artifact set under:
- `outputs/adult_census_income/`
- `data/processed/adult_census_income/`

Primary report-facing files:
- `outputs/adult_census_income/iteration2_steps_heading_fit.md`
- `outputs/adult_census_income/coursework_brief_6_steps_alignment.md`
- `outputs/adult_census_income/report_sections_draft.md`
